# 2. Network Security & Cloud Services Security Design

SC-100 asks: **"Design a network security architecture that implements Zero Trust and secures SaaS, PaaS, and IaaS workloads."**

## Setup

```bash
cd security/sc-100/03-infrastructure
uv sync
uv run python -m ipykernel install --user --name=sc-100 --display-name="SC-100 (Python)"
```
Then select the **SC-100 (Python)** kernel in VS Code's kernel picker (top-right).

## Glossary (read this first if you're new)

Network & services security is full of jargon. Here's plain-English meaning for the acronyms in this notebook.

| Term | What it means (plain English) |
|------|-------------------------------|
| **VNet** | *Virtual Network* — your private network inside Azure. Like a LAN in the cloud. |
| **Hub-spoke** | A topology where one central VNet (the *hub*) holds shared services (firewall, DNS, VPN) and other VNets (*spokes*) connect through it. |
| **NSG** | *Network Security Group* — a simple stateful firewall attached to a subnet or NIC (allow/deny rules). |
| **ASG** | *Application Security Group* — a label you attach to VMs so you can write NSG rules like "web → db" instead of IP ranges. |
| **Private Endpoint** | A private IP **inside your VNet** that points to a PaaS service (e.g., Azure SQL). The service's public IP is effectively turned off. |
| **Service Endpoint** | Older feature. Traffic to PaaS still hits its **public** IP but is optimized/whitelisted. Prefer Private Endpoint when possible. |
| **WAF** | *Web Application Firewall* — filters HTTP traffic against OWASP Top-10 rules (SQL injection, XSS, …). |
| **DDoS Protection** | Mitigates volumetric/L3-L4 floods against your public IPs. Two SKUs: *Network* (VNet-wide) and *IP Protection* (per public IP). |
| **SSE** | *Security Service Edge* — cloud-delivered SWG + ZTNA. Microsoft's implementation is **Entra Internet Access + Entra Private Access**. |
| **SWG** | *Secure Web Gateway* — filters outbound internet traffic by URL/category/threat. |
| **ZTNA** | *Zero Trust Network Access* — per-app access verified by identity instead of handing out full network access like a VPN. |
| **Azure Bastion** | Managed jump host: browser-based RDP/SSH to VMs that have **no public IP**. |
| **AKS** | *Azure Kubernetes Service* — managed Kubernetes. |
| **ACR** | *Azure Container Registry* — private Docker registry. |
| **CSI driver** | Kubernetes storage plug-in. The **Key Vault CSI driver** mounts secrets from Key Vault into pods. |
| **MCSB** | *Microsoft Cloud Security Benchmark* — recommended baseline, shipped as an Azure Policy initiative. |
| **Trusted Launch / Confidential VM** | Azure VM features: Secure Boot + vTPM (Trusted Launch), and memory encryption (Confidential) for sensitive workloads. |


## Network Segmentation Design

### Zero Trust networking: "Never trust, always verify" applies to the network too

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                     NETWORK SEGMENTATION ARCHITECTURE                       │
│                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────┐ │
│  │  LAYER 1: Subscription/VNet level (macro-segmentation)                 │ │
│  │  • Separate VNets for: production, staging, dev, shared services       │ │
│  │  • Hub-spoke topology with Azure Firewall in hub                       │ │
│  │  • VNet peering with controlled routes                                 │ │
│  └─────────────────────────────────────────────────────────────────────────┘ │
│                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────┐ │
│  │  LAYER 2: Subnet level (micro-segmentation)                            │ │
│  │  • NSGs on every subnet (default deny inbound)                         │ │
│  │  • Application Security Groups for role-based rules                    │ │
│  │  • Dedicated subnets: GatewaySubnet, AzureFirewallSubnet,             │ │
│  │    AzureBastionSubnet, PrivateEndpoints                                │ │
│  └─────────────────────────────────────────────────────────────────────────┘ │
│                                                                              │
│  ┌─────────────────────────────────────────────────────────────────────────┐ │
│  │  LAYER 3: Application level (identity-based access)                    │ │
│  │  • Private Endpoints for all PaaS services                             │ │
│  │  • Service Endpoints where Private Endpoints aren't available          │ │
│  │  • Azure Firewall for east-west traffic inspection                     │ │
│  │  • Entra Internet Access for identity-aware SWG                        │ │
│  └─────────────────────────────────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────────────────────────────────┘
```

## Bad practice → Best practice (network & services)

Each row is a recurring SC-100 exam pattern.

| ❌ Bad / legacy | ✅ Best practice | Why |
|-----------------|------------------|-----|
| Flat VNet where every spoke can talk to every spoke | **Hub-spoke with Azure Firewall in the hub + deny-by-default between spokes** | Limits lateral movement / blast radius |
| Azure SQL / Storage with public endpoint + firewall rules | **Private Endpoints + disable public network access** | Removes the public attack surface entirely |
| Public IPs + "just allow RDP from corp" | **Azure Bastion** (no public IPs) and/or **JIT VM access** | No exposed management ports |
| VPN for all remote users, full-tunnel into the network | **Entra Private Access (ZTNA)** per-app | Users get apps, not networks; ties into Conditional Access |
| Outbound internet through a legacy on-prem proxy | **Entra Internet Access (SWG)** | Cloud-scale, identity-aware, M365 token-protected |
| Detect misconfigurations after deployment | **Azure Policy "deny" effects at management-group scope** | Prevention beats detection; developers can't ship the mistake |
| WAF only in front of one app | **Front Door + WAF for global apps, App Gateway + WAF for regional** | Layered, global DDoS-L7 + OWASP rules everywhere |
| K8s Secrets (base64) | **Azure Key Vault + CSI driver + workload identity** | Real encryption, auditable, no pod-baked secrets |
| AKS with local accounts + public API server | **Private cluster + Entra ID + Azure RBAC** | No internet control plane, identity-based K8s perms |
| Azure OpenAI with API keys in code | **Managed identity + Private Endpoint + content filtering** | No secrets to leak; no public data path |
| "We'll harden it later" custom VM images | **MCSB + service baselines enforced by Azure Policy + Trusted Launch VMs** | Secure-by-default from day zero |
| Standard Azure VM for sensitive workloads | **Trusted Launch (Secure Boot + vTPM)** and **Confidential VMs** for PHI/PCI data | Boot-chain integrity and memory encryption |


In [1]:
import json

# ===================================================================
# NETWORK SECURITY TOOL SELECTION
# Architects must pick the right tool for each network security need
# ===================================================================

NETWORK_TOOLS = [
    {
        'need': 'Filter east-west traffic between VNets',
        'tool': 'Azure Firewall (Premium)',
        'why': 'Centralized L4-L7 firewall with TLS inspection, IDPS, URL filtering',
        'alternatives': 'NVA (Palo Alto, Fortinet) if specific vendor features needed',
    },
    {
        'need': 'Protect web applications from OWASP Top 10',
        'tool': 'Azure WAF on Application Gateway or Front Door',
        'why': 'WAF rules (OWASP 3.2 rule set), bot protection, geo-filtering',
        'alternatives': 'Azure WAF on CDN for simpler scenarios',
    },
    {
        'need': 'DDoS protection for public endpoints',
        'tool': 'Azure DDoS Protection (Standard)',
        'why': 'Adaptive tuning, mitigation reports, cost protection guarantee',
        'alternatives': 'DDoS IP Protection for individual IPs (cheaper)',
    },
    {
        'need': 'Secure access to PaaS services (SQL, Storage)',
        'tool': 'Private Endpoints',
        'why': 'PaaS service gets a private IP in your VNet — no public internet exposure',
        'alternatives': 'Service Endpoints (less secure — traffic still goes to public endpoint)',
    },
    {
        'need': 'Secure remote access to VMs (replace VPN jump boxes)',
        'tool': 'Azure Bastion',
        'why': 'Browser-based RDP/SSH, no public IP on VMs, Entra ID auth support',
        'alternatives': 'JIT VM access (Defender for Cloud) to temporarily open RDP/SSH port',
    },
    {
        'need': 'Identity-aware internet access (replace traditional SWG)',
        'tool': 'Microsoft Entra Internet Access',
        'why': 'Part of SSE — integrates with Conditional Access, user/group aware filtering',
        'alternatives': 'Zscaler, Netskope (3rd party SSE)',
    },
    {
        'need': 'Replace VPN for private app access',
        'tool': 'Microsoft Entra Private Access',
        'why': 'ZTNA (Zero Trust Network Access) — identity-verified access to private apps',
        'alternatives': 'Azure VPN Gateway or ExpressRoute (traditional approach)',
    },
    {
        'need': 'Monitor network traffic for threats',
        'tool': 'NSG Flow Logs + Traffic Analytics',
        'why': 'Visualize traffic patterns, detect anomalies, identify unused NSG rules',
        'alternatives': 'Azure Firewall logs + Sentinel for deeper analysis',
    },
]

print('=== Network Security Tool Selection Guide ===\n')
print(f'{"Need":<55} {"Primary Tool":<45} {"Alternative"}')
print('─' * 140)
for tool in NETWORK_TOOLS:
    print(f'{tool["need"]:<55} {tool["tool"]:<45} {tool["alternatives"]}')

=== Network Security Tool Selection Guide ===

Need                                                    Primary Tool                                  Alternative
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Filter east-west traffic between VNets                  Azure Firewall (Premium)                      NVA (Palo Alto, Fortinet) if specific vendor features needed
Protect web applications from OWASP Top 10              Azure WAF on Application Gateway or Front Door Azure WAF on CDN for simpler scenarios
DDoS protection for public endpoints                    Azure DDoS Protection (Standard)              DDoS IP Protection for individual IPs (cheaper)
Secure access to PaaS services (SQL, Storage)           Private Endpoints                             Service Endpoints (less secure — traffic still goes to public endpoint)
Secure remote access to VMs (replace VPN jump boxes)    Azure Bastion

## Microsoft SSE (Security Service Edge)

SSE is Microsoft's answer to secure access in a hybrid work world — replacing VPN + proxy + SWG.

```
┌──────────────────────────────────────────────────────────────────────┐
│              MICROSOFT SECURITY SERVICE EDGE (SSE)                  │
│                                                                      │
│  ┌─────────────────────┐  ┌─────────────────────────────────────┐   │
│  │  Entra Internet     │  │  Entra Private Access               │   │
│  │  Access              │  │                                     │   │
│  │                     │  │  ZTNA (Zero Trust Network Access)   │   │
│  │  • Secure Web       │  │  • Replace VPN for private apps     │   │
│  │    Gateway (SWG)    │  │  • Per-app access (not network)     │   │
│  │  • Web category     │  │  • Conditional Access integrated    │   │
│  │    filtering        │  │  • No network-level access          │   │
│  │  • Threat protection│  │  • App connectors on-prem           │   │
│  │  • TLS inspection   │  │                                     │   │
│  │  • Universal CA     │  │  User → Entra ID → App Connector   │   │
│  │    (token protection│  │         → Private App               │   │
│  │    for M365)        │  │                                     │   │
│  └─────────────────────┘  └─────────────────────────────────────┘   │
│                                                                      │
│  Client: Global Secure Access client (Windows, macOS, iOS, Android) │
│  Integration: Conditional Access, Defender for Cloud Apps, DLP       │
└──────────────────────────────────────────────────────────────────────┘
```

### SSE vs traditional approaches:

| Approach | SSE (Entra Internet/Private Access) | Traditional (VPN + Proxy) |
|----------|--------------------------------------|---------------------------|
| Access model | Per-app, identity-based | Network-level (full tunnel) |
| MFA integration | Native Conditional Access | Separate MFA for VPN |
| User experience | Always-on, transparent | Connect/disconnect VPN |
| Scalability | Microsoft global network | VPN concentrator capacity |
| Lateral movement risk | Low (app-level isolation) | High (network access = everything) |
| M365 optimization | Token protection, direct routing | Split tunnel or backhaul |

In [2]:
# ===================================================================
# SCENARIO: Design network security for Tailspin Toys
# ===================================================================

SCENARIO = """
COMPANY: Tailspin Toys (e-commerce retailer)
EMPLOYEES: 3,000 (60% hybrid/remote workers)
CUSTOMERS: 2 million online shoppers

INFRASTRUCTURE:
  Azure:
    - Hub-spoke network (1 hub VNet, 4 spoke VNets)
    - AKS cluster running microservices (customer-facing)
    - Azure SQL with customer data + payment info (PCI DSS scope)
    - Azure Front Door for global load balancing
    - 20 Azure Functions for backend processing

  On-premises:
    - Warehouse management system (legacy app on Windows Server)
    - VPN concentrator for remote employee access
    - Corporate Wi-Fi with captive portal

CURRENT PROBLEMS:
  - VPN overloaded during peak hours (remote workers can't connect)
  - Web app received DDoS attack last Black Friday
  - Developer accidentally exposed Azure SQL to internet (public endpoint)
  - No east-west traffic filtering (all spoke VNets can talk to each other)
  - Legacy warehouse app accessible from any internal network segment

GOALS:
  - PCI DSS compliance for payment processing
  - Replace VPN with modern remote access
  - Protect customer-facing web app from attacks
  - Segment network to limit blast radius
"""

print(SCENARIO)


COMPANY: Tailspin Toys (e-commerce retailer)
EMPLOYEES: 3,000 (60% hybrid/remote workers)
CUSTOMERS: 2 million online shoppers

INFRASTRUCTURE:
  Azure:
    - Hub-spoke network (1 hub VNet, 4 spoke VNets)
    - AKS cluster running microservices (customer-facing)
    - Azure SQL with customer data + payment info (PCI DSS scope)
    - Azure Front Door for global load balancing
    - 20 Azure Functions for backend processing

  On-premises:
    - Warehouse management system (legacy app on Windows Server)
    - VPN concentrator for remote employee access
    - Corporate Wi-Fi with captive portal

CURRENT PROBLEMS:
  - VPN overloaded during peak hours (remote workers can't connect)
  - Web app received DDoS attack last Black Friday
  - Developer accidentally exposed Azure SQL to internet (public endpoint)
  - No east-west traffic filtering (all spoke VNets can talk to each other)
  - Legacy warehouse app accessible from any internal network segment

GOALS:
  - PCI DSS compliance for paymen

In [3]:
# ===================================================================
# NETWORK SECURITY ARCHITECTURE DESIGN
# ===================================================================

NETWORK_DESIGN = {
    'Perimeter protection': [
        {'component': 'Azure Front Door + WAF', 'purpose': 'Global LB, DDoS L7, OWASP rule set, geo-filtering', 'pci': 'Req 6.6'},
        {'component': 'Azure DDoS Protection Standard', 'purpose': 'L3/L4 DDoS mitigation with adaptive tuning', 'pci': 'N/A'},
        {'component': 'Azure Firewall Premium (hub)', 'purpose': 'East-west filtering, TLS inspection, IDPS', 'pci': 'Req 1.3'},
    ],
    'Segmentation': [
        {'component': 'Spoke VNet: Web tier', 'purpose': 'AKS cluster (customer-facing)', 'pci': 'Req 1.2'},
        {'component': 'Spoke VNet: App tier', 'purpose': 'Azure Functions + internal APIs', 'pci': 'Req 1.2'},
        {'component': 'Spoke VNet: Data tier (CDE)', 'purpose': 'Azure SQL + Storage (PCI scope)', 'pci': 'Req 1.3.6'},
        {'component': 'Spoke VNet: Management', 'purpose': 'Bastion, monitoring, admin tools', 'pci': 'Req 7.1'},
        {'component': 'NSGs on all subnets', 'purpose': 'Default deny, allow only required flows', 'pci': 'Req 1.2'},
    ],
    'PaaS security': [
        {'component': 'Private Endpoints for Azure SQL', 'purpose': 'Remove public endpoint entirely', 'pci': 'Req 1.3.2'},
        {'component': 'Private Endpoints for Storage', 'purpose': 'No public blob access', 'pci': 'Req 1.3.2'},
        {'component': 'Azure Policy: deny public endpoints', 'purpose': 'Prevent developers from exposing PaaS', 'pci': 'Req 2.2'},
    ],
    'Remote access (replace VPN)': [
        {'component': 'Entra Private Access', 'purpose': 'ZTNA to warehouse app and internal tools', 'pci': 'N/A'},
        {'component': 'Entra Internet Access', 'purpose': 'SWG for remote worker internet traffic', 'pci': 'N/A'},
        {'component': 'Azure Bastion', 'purpose': 'Admin access to VMs (no public IPs)', 'pci': 'Req 8.3'},
    ],
}

print('=== Network Security Architecture ===\n')
for category, components in NETWORK_DESIGN.items():
    print(f'\n{"=" * 80}')
    print(f'{category}')
    print(f'{"=" * 80}')
    print(f'{"Component":<40} {"Purpose":<50} {"PCI DSS"}')
    print('─' * 100)
    for c in components:
        print(f'{c["component"]:<40} {c["purpose"]:<50} {c["pci"]}')

=== Network Security Architecture ===


Perimeter protection
Component                                Purpose                                            PCI DSS
────────────────────────────────────────────────────────────────────────────────────────────────────
Azure Front Door + WAF                   Global LB, DDoS L7, OWASP rule set, geo-filtering  Req 6.6
Azure DDoS Protection Standard           L3/L4 DDoS mitigation with adaptive tuning         N/A
Azure Firewall Premium (hub)             East-west filtering, TLS inspection, IDPS          Req 1.3

Segmentation
Component                                Purpose                                            PCI DSS
────────────────────────────────────────────────────────────────────────────────────────────────────
Spoke VNet: Web tier                     AKS cluster (customer-facing)                      Req 1.2
Spoke VNet: App tier                     Azure Functions + internal APIs                    Req 1.2
Spoke VNet: Data tier (CDE)

In [4]:
# ===================================================================
# PRACTICAL EXAMPLE: Validate an NSG rule set for a 3-tier app
# Beginner-friendly check that your proposed rules follow Zero Trust:
#   - default deny inbound
#   - no 0.0.0.0/0 on management ports (RDP 3389, SSH 22)
#   - DB tier never reachable from the internet
# ===================================================================

from dataclasses import dataclass

@dataclass
class NsgRule:
    name: str
    priority: int
    direction: str      # 'Inbound' or 'Outbound'
    access: str         # 'Allow' or 'Deny'
    source: str         # CIDR or tag like 'Internet', 'VirtualNetwork'
    dest: str           # subnet tag like 'web', 'app', 'db'
    port: str           # '443', '3389', '1433', '*'

# Proposed rules for Tailspin's web subnet
web_subnet_rules = [
    NsgRule('AllowHttpsIn',     100, 'Inbound',  'Allow', 'Internet',       'web', '443'),
    NsgRule('AllowAppFromWeb',  110, 'Inbound',  'Allow', 'web',            'app', '8080'),
    NsgRule('AllowDbFromApp',   120, 'Inbound',  'Allow', 'app',            'db',  '1433'),
    NsgRule('DenyAllInbound',  4096, 'Inbound',  'Deny',  '*',              '*',   '*'),
]

def audit(rules):
    findings = []
    has_default_deny = any(r.access == 'Deny' and r.source == '*' and r.direction == 'Inbound'
                           for r in rules)
    if not has_default_deny:
        findings.append('MISSING default-deny inbound rule (Zero Trust baseline)')

    for r in rules:
        if r.access == 'Allow' and r.source == 'Internet' and r.port in ('3389', '22'):
            findings.append(f'{r.name}: allows {r.port} from Internet — use Bastion + JIT instead')
        if r.access == 'Allow' and r.source == 'Internet' and r.dest == 'db':
            findings.append(f'{r.name}: exposes DB tier to Internet')
    return findings

issues = audit(web_subnet_rules)
if not issues:
    print('✅ NSG rule set passes Zero Trust baseline checks')
else:
    print('❌ Findings:')
    for f in issues:
        print(f'  - {f}')

# Try a bad rule set to see the audit catch it:
print()
print('--- Running audit on a deliberately bad rule set ---')
bad = [
    NsgRule('AllowRdpIn', 100, 'Inbound', 'Allow', 'Internet', 'web', '3389'),
    NsgRule('AllowSqlIn', 110, 'Inbound', 'Allow', 'Internet', 'db',  '1433'),
]
for f in audit(bad):
    print(f'  - {f}')


✅ NSG rule set passes Zero Trust baseline checks

--- Running audit on a deliberately bad rule set ---
  - MISSING default-deny inbound rule (Zero Trust baseline)
  - AllowRdpIn: allows 3389 from Internet — use Bastion + JIT instead
  - AllowSqlIn: exposes DB tier to Internet


## Container & AKS Security Design

### Defense in depth for containerized workloads:

```
┌──────────────────────────────────────────────────────────────────────┐
│                    CONTAINER SECURITY LAYERS                         │
│                                                                      │
│  1. IMAGE SECURITY (build time)                                     │
│     • Defender for Containers: image vulnerability scanning          │
│     • ACR (Azure Container Registry) with content trust              │
│     • Base image selection: Microsoft Artifact Registry (MAR)        │
│     • No secrets in images — use Key Vault CSI driver                │
│                                                                      │
│  2. REGISTRY SECURITY                                                │
│     • ACR with Private Endpoint (no public access)                   │
│     • ACR firewall rules (allow only AKS subnet)                     │
│     • Image quarantine until scan passes                             │
│     • Geo-replication for availability                               │
│                                                                      │
│  3. CLUSTER SECURITY (AKS)                                           │
│     • Entra ID integration for RBAC (no local accounts)              │
│     • Azure Policy for Kubernetes (OPA Gatekeeper)                   │
│     • Private cluster (API server not exposed to internet)           │
│     • Network policies (Calico/Azure CNI) for pod-to-pod             │
│     • Workload identity (no service account tokens)                  │
│                                                                      │
│  4. RUNTIME SECURITY                                                 │
│     • Defender for Containers runtime protection                     │
│     • Pod Security Standards (restricted profile)                    │
│     • Immutable containers (read-only root filesystem)               │
│     • Resource limits and quotas                                     │
└──────────────────────────────────────────────────────────────────────┘
```

In [5]:
# ===================================================================
# CONTAINER SECURITY DESIGN DECISIONS
# ===================================================================

AKS_SECURITY_DECISIONS = [
    {
        'decision': 'AKS authentication method',
        'options': ['Local accounts', 'Entra ID integration', 'Entra ID + Azure RBAC'],
        'recommended': 'Entra ID + Azure RBAC',
        'reason': 'Disable local accounts entirely. Use Entra ID groups for namespace RBAC. '
                  'Azure RBAC provides granular Kubernetes permissions via Azure role assignments.',
    },
    {
        'decision': 'AKS network model',
        'options': ['kubenet', 'Azure CNI', 'Azure CNI Overlay', 'Azure CNI + Cilium'],
        'recommended': 'Azure CNI Overlay (or CNI + Cilium for advanced)',
        'reason': 'CNI Overlay gives pods routable IPs without consuming VNet IPs. '
                  'Cilium adds eBPF-based network policies and observability.',
    },
    {
        'decision': 'Secrets management for pods',
        'options': ['Kubernetes Secrets (base64)', 'Key Vault CSI driver', 'External Secrets Operator'],
        'recommended': 'Key Vault CSI driver',
        'reason': 'Kubernetes Secrets are base64 encoded (not encrypted). Key Vault CSI driver '
                  'mounts Key Vault secrets directly into pods. Use with workload identity.',
    },
    {
        'decision': 'Container image source policy',
        'options': ['Allow any registry', 'ACR only', 'ACR + validated MAR images'],
        'recommended': 'ACR + validated MAR images',
        'reason': 'Azure Policy for Kubernetes can enforce that only images from your ACR are allowed. '
                  'Import base images from Microsoft Artifact Registry into ACR for scanning.',
    },
    {
        'decision': 'API server exposure',
        'options': ['Public API server', 'API server with authorized IP ranges', 'Private cluster'],
        'recommended': 'Private cluster',
        'reason': 'Private cluster puts API server on private VNet. Access via Bastion, VPN, '
                  'or Entra Private Access. No internet exposure of control plane.',
    },
]

print('=== AKS Security Design Decisions ===\n')
for d in AKS_SECURITY_DECISIONS:
    print(f'\n--- {d["decision"]} ---')
    print(f'  Options: {", ".join(d["options"])}')
    print(f'  Recommended: {d["recommended"]}')
    print(f'  Reason: {d["reason"]}')

=== AKS Security Design Decisions ===


--- AKS authentication method ---
  Options: Local accounts, Entra ID integration, Entra ID + Azure RBAC
  Recommended: Entra ID + Azure RBAC
  Reason: Disable local accounts entirely. Use Entra ID groups for namespace RBAC. Azure RBAC provides granular Kubernetes permissions via Azure role assignments.

--- AKS network model ---
  Options: kubenet, Azure CNI, Azure CNI Overlay, Azure CNI + Cilium
  Recommended: Azure CNI Overlay (or CNI + Cilium for advanced)
  Reason: CNI Overlay gives pods routable IPs without consuming VNet IPs. Cilium adds eBPF-based network policies and observability.

--- Secrets management for pods ---
  Options: Kubernetes Secrets (base64), Key Vault CSI driver, External Secrets Operator
  Recommended: Key Vault CSI driver
  Reason: Kubernetes Secrets are base64 encoded (not encrypted). Key Vault CSI driver mounts Key Vault secrets directly into pods. Use with workload identity.

--- Container image source policy ---
  O

In [6]:
# ===================================================================
# AI SERVICES SECURITY DESIGN
# Hot topic: securing Azure OpenAI and AI workloads
# ===================================================================

AI_SECURITY_DESIGN = {
    'Azure OpenAI Security': [
        {'control': 'Network isolation', 'implementation': 'Private Endpoint + disable public access', 'risk': 'Data exfiltration via API'},
        {'control': 'Authentication', 'implementation': 'Managed identity (not API keys)', 'risk': 'API key leakage'},
        {'control': 'Data residency', 'implementation': 'Deploy in specific region, data stays in region', 'risk': 'Regulatory violation'},
        {'control': 'Content filtering', 'implementation': 'Azure AI Content Safety (built-in)', 'risk': 'Harmful content generation'},
        {'control': 'Prompt injection defense', 'implementation': 'Input validation + system message guardrails', 'risk': 'Prompt injection attacks'},
        {'control': 'Usage monitoring', 'implementation': 'Diagnostic logs → Sentinel, abuse detection', 'risk': 'Misuse or abuse'},
        {'control': 'RBAC', 'implementation': 'Separate roles: Cognitive Services User vs Contributor', 'risk': 'Unauthorized model deployment'},
    ],
    'Copilot for M365 Security': [
        {'control': 'Data access', 'implementation': 'Copilot respects existing M365 permissions (SharePoint, OneDrive)', 'risk': 'Over-permissioned access'},
        {'control': 'Oversharing prevention', 'implementation': 'Review SharePoint permissions BEFORE Copilot deployment', 'risk': 'Copilot surfaces data users shouldn\'t see'},
        {'control': 'Sensitivity labels', 'implementation': 'Purview labels honored by Copilot', 'risk': 'Confidential data in Copilot responses'},
        {'control': 'DLP', 'implementation': 'Purview DLP applies to Copilot-generated content', 'risk': 'Data leakage through Copilot'},
        {'control': 'Audit', 'implementation': 'Copilot interactions logged in Purview Audit', 'risk': 'Compliance gaps'},
    ],
}

print('=== AI Services Security Design ===\n')
for service, controls in AI_SECURITY_DESIGN.items():
    print(f'\n{"=" * 80}')
    print(f'{service}')
    print(f'{"=" * 80}')
    print(f'{"Control":<25} {"Implementation":<55} {"Risk Mitigated"}')
    print('─' * 110)
    for c in controls:
        print(f'{c["control"]:<25} {c["implementation"]:<55} {c["risk"]}')

=== AI Services Security Design ===


Azure OpenAI Security
Control                   Implementation                                          Risk Mitigated
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
Network isolation         Private Endpoint + disable public access                Data exfiltration via API
Authentication            Managed identity (not API keys)                         API key leakage
Data residency            Deploy in specific region, data stays in region         Regulatory violation
Content filtering         Azure AI Content Safety (built-in)                      Harmful content generation
Prompt injection defense  Input validation + system message guardrails            Prompt injection attacks
Usage monitoring          Diagnostic logs → Sentinel, abuse detection             Misuse or abuse
RBAC                      Separate roles: Cognitive Services User vs Contributor  Unauthorized model deployme

In [7]:
# ===================================================================
# SECURITY BASELINES DESIGN
# ===================================================================

SECURITY_BASELINES = {
    'Microsoft Cloud Security Benchmark (MCSB)': {
        'scope': 'All Azure services',
        'enforcement': 'Azure Policy initiatives mapped to MCSB controls',
        'key_areas': ['Network security', 'Identity management', 'Data protection',
                      'Logging and threat detection', 'Posture and vulnerability management'],
        'design_tip': 'Assign MCSB policy initiative at management group level for org-wide compliance',
    },
    'Service-specific baselines': {
        'scope': 'Individual Azure services',
        'examples': [
            'Azure SQL: TDE enabled, AAD-only auth, private endpoint, auditing to Sentinel',
            'Storage: HTTPS only, minimum TLS 1.2, private endpoint, soft delete, versioning',
            'AKS: Entra ID auth, private cluster, network policies, Defender for Containers',
            'Key Vault: RBAC (not access policies), private endpoint, soft delete + purge protection',
        ],
        'design_tip': 'Create custom Azure Policy initiatives per service type for consistent baseline enforcement',
    },
    'CIS Benchmarks': {
        'scope': 'OS-level (Windows, Linux)',
        'enforcement': 'Guest Configuration (Azure Policy) or Intune security baselines',
        'key_areas': ['Account policies', 'Audit policies', 'Security options', 'Firewall'],
        'design_tip': 'Use Azure Automanage for automated CIS baseline compliance on VMs',
    },
}

print('=== Security Baselines Architecture ===\n')
for baseline, details in SECURITY_BASELINES.items():
    print(f'\n--- {baseline} ---')
    print(f'  Scope: {details["scope"]}')
    if 'enforcement' in details:
        print(f'  Enforcement: {details["enforcement"]}')
    if 'key_areas' in details:
        print(f'  Key areas: {", ".join(details["key_areas"])}')
    if 'examples' in details:
        print(f'  Examples:')
        for ex in details['examples']:
            print(f'    • {ex}')
    print(f'  Design tip: {details["design_tip"]}')

=== Security Baselines Architecture ===


--- Microsoft Cloud Security Benchmark (MCSB) ---
  Scope: All Azure services
  Enforcement: Azure Policy initiatives mapped to MCSB controls
  Key areas: Network security, Identity management, Data protection, Logging and threat detection, Posture and vulnerability management
  Design tip: Assign MCSB policy initiative at management group level for org-wide compliance

--- Service-specific baselines ---
  Scope: Individual Azure services
  Examples:
    • Azure SQL: TDE enabled, AAD-only auth, private endpoint, auditing to Sentinel
    • Storage: HTTPS only, minimum TLS 1.2, private endpoint, soft delete, versioning
    • AKS: Entra ID auth, private cluster, network policies, Defender for Containers
    • Key Vault: RBAC (not access policies), private endpoint, soft delete + purge protection
  Design tip: Create custom Azure Policy initiatives per service type for consistent baseline enforcement

--- CIS Benchmarks ---
  Scope: OS-level (Windo

## Practical example: prevent public Azure SQL with Azure Policy

This is the *preventive* control from the quiz (deny instead of detect). In real life you'd assign
an existing built-in policy, but the JSON below shows exactly what that policy does — useful when
you need to understand what you're turning on, or to build a custom variant.


In [8]:
# ===================================================================
# PRACTICAL EXAMPLE: Azure Policy definition that DENIES creating an
# Azure SQL server with public network access. This is the preventive
# control referenced in the quiz.
#
# In production you'd assign the built-in policy "Azure SQL Database
# should have public network access disabled" — the JSON below shows
# the equivalent rule so you can read what such a policy looks like.
# ===================================================================

import json

deny_public_sql = {
  'properties': {
    'displayName': 'Deny Azure SQL servers with public network access',
    'policyType': 'Custom',
    'mode': 'Indexed',
    'parameters': {
      'effect': {
        'type': 'String',
        'allowedValues': ['Deny', 'Audit', 'Disabled'],
        'defaultValue': 'Deny',
      }
    },
    'policyRule': {
      'if': {
        'allOf': [
          {'field': 'type',
           'equals': 'Microsoft.Sql/servers'},
          {'field': 'Microsoft.Sql/servers/publicNetworkAccess',
           'notEquals': 'Disabled'}
        ]
      },
      'then': {'effect': "[parameters('effect')]"}
    }
  }
}

print(json.dumps(deny_public_sql, indent=2))
print()
print('Deploy with:')
print('  az policy definition create --name deny-public-sql \\')
print('     --rules "$(jq .properties.policyRule policy.json)" \\')
print('     --params "$(jq .properties.parameters policy.json)" \\')
print('     --mode Indexed')
print('  az policy assignment create --policy deny-public-sql \\')
print('     --scope /providers/Microsoft.Management/managementGroups/<mg-id>')


{
  "properties": {
    "displayName": "Deny Azure SQL servers with public network access",
    "policyType": "Custom",
    "mode": "Indexed",
    "parameters": {
      "effect": {
        "type": "String",
        "allowedValues": [
          "Deny",
          "Audit",
          "Disabled"
        ],
        "defaultValue": "Deny"
      }
    },
    "policyRule": {
      "if": {
        "allOf": [
          {
            "field": "type",
            "equals": "Microsoft.Sql/servers"
          },
          {
            "field": "Microsoft.Sql/servers/publicNetworkAccess",
            "notEquals": "Disabled"
          }
        ]
      },
      "then": {
        "effect": "[parameters('effect')]"
      }
    }
  }
}

Deploy with:
  az policy definition create --name deny-public-sql \
     --rules "$(jq .properties.policyRule policy.json)" \
     --params "$(jq .properties.parameters policy.json)" \
     --mode Indexed
  az policy assignment create --policy deny-public-sql \
     --scop

In [9]:
# ===================================================================
# ARCHITECTURE QUIZ: Network & Services Security
# ===================================================================

QUIZ = [
    {
        'question': 'Tailspin Toys needs to replace their overloaded VPN for 1,800 remote workers\n'
                    'accessing internal apps. Which solution provides the BEST Zero Trust approach?',
        'options': {
            'A': 'Azure VPN Gateway with point-to-site connections',
            'B': 'Microsoft Entra Private Access (ZTNA)',
            'C': 'Azure Bastion for all users',
            'D': 'ExpressRoute with private peering',
        },
        'answer': 'B',
        'explanation': 'Entra Private Access provides ZTNA — per-app access verified by identity, not '
                       'network-level access. Users get access to specific apps, not the entire network. '
                       'It integrates with Conditional Access for device compliance and risk checks. '
                       'VPN gives network-level access (not Zero Trust). Bastion is for admin access to VMs.',
    },
    {
        'question': 'A developer created an Azure SQL database with a public endpoint. How should\n'
                    'the architect PREVENT this from happening again organization-wide?',
        'options': {
            'A': 'Write a Sentinel analytics rule to detect public SQL endpoints',
            'B': 'Assign Azure Policy with "Deny" effect to block public network access on SQL',
            'C': 'Add a step in the CI/CD pipeline to check for public endpoints',
            'D': 'Enable Defender for Cloud recommendation for private endpoints',
        },
        'answer': 'B',
        'explanation': 'Azure Policy with "Deny" effect prevents the deployment at creation time — the '
                       'developer literally cannot create a public SQL endpoint. This is preventive control. '
                       'Sentinel and Defender for Cloud are detective (find after the fact). CI/CD check '
                       'only works for IaC-deployed resources, not portal/CLI deployments.',
    },
    {
        'question': 'Tailspin Toys is deploying Azure OpenAI for a customer-facing chatbot.\n'
                    'Which authentication method should the architect recommend?',
        'options': {
            'A': 'API key stored in Azure Key Vault',
            'B': 'Managed identity on the calling application',
            'C': 'OAuth 2.0 with client credentials flow',
            'D': 'Shared access signature (SAS) token',
        },
        'answer': 'B',
        'explanation': 'Managed identity eliminates credential management entirely — no secrets to rotate, '
                       'leak, or store. The calling app (App Service, AKS, Function) gets a managed identity '
                       'that is granted "Cognitive Services User" role on the Azure OpenAI resource. '
                       'API keys can leak. SAS tokens are for Storage, not Cognitive Services.',
    },
]

print('=== Network & Services Security Quiz ===\n')
for i, q in enumerate(QUIZ, 1):
    print(f'Question {i}:')
    print(f'{q["question"]}\n')
    for key, option in q['options'].items():
        marker = '>>>' if key == q['answer'] else '   '
        print(f'  {marker} {key}. {option}')
    print(f'\n  Answer: {q["answer"]}')
    print(f'  Why: {q["explanation"]}')
    print()

=== Network & Services Security Quiz ===

Question 1:
Tailspin Toys needs to replace their overloaded VPN for 1,800 remote workers
accessing internal apps. Which solution provides the BEST Zero Trust approach?

      A. Azure VPN Gateway with point-to-site connections
  >>> B. Microsoft Entra Private Access (ZTNA)
      C. Azure Bastion for all users
      D. ExpressRoute with private peering

  Answer: B
  Why: Entra Private Access provides ZTNA — per-app access verified by identity, not network-level access. Users get access to specific apps, not the entire network. It integrates with Conditional Access for device compliance and risk checks. VPN gives network-level access (not Zero Trust). Bastion is for admin access to VMs.

Question 2:
A developer created an Azure SQL database with a public endpoint. How should
the architect PREVENT this from happening again organization-wide?

      A. Write a Sentinel analytics rule to detect public SQL endpoints
  >>> B. Assign Azure Policy with